# **Smart Water Systems: Regression Models for Time-Series Forecasting**

## Learning goals
By the end of this notebook, students should be able to:

1. Use **ordinary least squares (OLS)** regression to model water demand.
2. Distinguish between **external predictors**, **lagged predictors**, and **categorical time predictors**.
3. Create lagged variables from a time series using `shift()`.
4. Represent time of day as a categorical variable using dummy variables.
5. Fit a regression model with `LinearRegression()` and generate predictions with `predict()`.
6. Evaluate predictions using MAE and RMSE.
7. Compare observed and predicted demand in both scatterplots and time-series plots.
8. Build a direct multi-step OLS forecast with a separate model for each forecast horizon.

We will use **DMA 1 demand (L/s)** and **air temperature (°C)** throughout the notebook.


# **Regression forecasting: the basic idea**

In regression, we describe the target variable $y_t$ using one or more predictors $x_t$:

$$
y_t = \beta_0 + \beta_1x_{1,t} + \beta_2x_{2,t}+\cdots+\beta_px_{p,t}+\varepsilon_t
$$

where:

- $y_t$ is the observed demand at time $t$,
- $x_{j,t}$ are predictor variables,
- $\beta_0$ is the intercept,
- $\beta_j$ are coefficients estimated from the data, and
- $\varepsilon_t$ is the part of demand not explained by the model.

**OLS** chooses the coefficients that minimize the sum of squared residuals:

$$
\sum_t (y_t-\hat y_t)^2
$$

We will build three models so we can see how different types of predictors enter a regression:

1. **Temperature model**: an external predictor
2. **Lag model**: previous values of the demand time series
3. **Temperature + time-of-day model**: a numerical predictor plus a categorical predictor

The purpose is not simply to find the most accurate model. The goal is to understand how each type of information is represented in a forecasting model.


In [ ]:
# imprort packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# **1. Load the DMA 1 data**

The CSV file contains timestamps, water demand, and air temperature. The temperature variable has already been added to the file, so we will read it directly rather than generate it in the notebook.


In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

# Define course folders
course_folder = '/content/drive/MyDrive/CE374U2_Fall2026'
data_folder = os.path.join(course_folder, 'data')
figures_folder = os.path.join(course_folder, 'figures')

file_name = '02_04_timeseries_forecast.csv'
data_file = os.path.join(data_folder, file_name)

data_file


In [ ]:
df = pd.read_csv(data_file)
df.head()


### **Rename and select the variables**

Long column names are useful in a data file, but shorter names make Python code easier to read.

`rename()` changes the column labels. It does **not** change the data.

`pd.to_datetime()` converts the timestamp from text into a datetime object. The argument `dayfirst=True` is important because these data use the European date format **day/month/year**.

Finally, `df[['timestamp', 'DMA1', 'Temperature']]` keeps only the columns needed in this notebook.

Notice the **double brackets**:

```python
df[['timestamp', 'DMA1']]
```

Double brackets return a DataFrame containing multiple selected columns. A single bracket such as `df['DMA1']` returns one Series.


In [ ]:
# rename columns

df = df.rename(columns={
    'Date-time CET-CEST (DD/MM/YYYY HH:mm)': 'timestamp',
    'DMA 1 (L/s)': 'DMA1',
    'Air temperature (C)': 'Temperature'
})

# convert to datetime

df['timestamp'] = pd.to_datetime(df['timestamp'], dayfirst=True)

df = df[['timestamp', 'DMA1', 'Temperature']]

df.head()


### **Check the data**

Before building a model, check the period of record and missing values. A regression model cannot be fit to rows in which the target or a required predictor is missing.


In [ ]:
print('Start:', df['timestamp'].min())
print('End:', df['timestamp'].max())
print('Number of records:', len(df))
print('Missing DMA1 values:', df['DMA1'].isna().sum())
print('Missing temperature values:', df['Temperature'].isna().sum())


# **2. Explore demand and temperature**

Always inspect the time series before fitting a model. Water demand often contains strong daily patterns, while temperature changes more gradually.


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['timestamp'], df['DMA1'])
plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('DMA 1 Demand')
plt.grid(True)
plt.show()


### Zoom in on two weeks

A shorter window makes the daily demand cycle easier to see.


In [ ]:
start = '2023-03-01'
end = '2023-03-15'

subset = df[(df['timestamp'] >= start) & (df['timestamp'] < end)]

plt.figure(figsize=(12, 4))
plt.plot(subset['timestamp'], subset['DMA1'])
plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('DMA 1 Demand: Two-Week Example')
plt.grid(True)
plt.show()


# **3. Create training and testing periods**

We should not evaluate a model on the same observations used to estimate its coefficients.

For time-series data, the split must preserve time order:

- **training data** come first and are used to estimate the model coefficients;
- **testing data** come later and are used to evaluate predictions.

Here, the final 14 days are reserved for testing.

### What does `.copy()` do?

You will see code such as:

```python
train = df[df['timestamp'] < test_start].copy()
```

The expression inside the brackets selects rows. `.copy()` then creates a separate DataFrame containing those rows. This is useful because we will later add columns such as predictions. Using a copy avoids unintentionally changing the original `df` DataFrame.


In [ ]:
# from pandas._libs.tslibs import timestamps

# test start date
test_start = '2023-03-15'

# train data
train = df[df['timestamp'] < test_start].copy()

# test data
test = df[df['timestamp'] >= test_start].copy()

## if we want to select a specific period
# test_end = df['timestamp'].max()
# test = df[
#     (df['timestamp'] >= test_start) &
#     (df['timestamp'] <= test_end)
# ].copy()

print('Training period:', train['timestamp'].min(), 'to', train['timestamp'].max())
print('Testing period:', test['timestamp'].min(), 'to', test['timestamp'].max())
print('Testing hours:', len(test))


In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
# plot train and test data

plt.figure(figsize=(12, 4))
plt.plot(train['timestamp'], train['DMA1'], label = 'Train')
plt.plot(test['timestamp'], test['DMA1'], label = 'Test')
plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('DMA 1 Demand')
plt.legend()
plt.grid(True)
plt.show()


# **4. Model 1: OLS using temperature**

We begin with one external predictor: **air temperature**.

The model is

$$
D_t = \beta_0 + \beta_1 T_t + \varepsilon_t
$$

where:

- $D_t$ = DMA 1 demand at hour $t$,
- $T_t$ = air temperature at hour $t$,
- $\beta_0$ = intercept,
- $\beta_1$ = change in expected demand associated with a 1°C change in temperature.

The fitted model is

$$
\hat D_t = \hat\beta_0 + \hat\beta_1 T_t
$$

## **Preparing the data**

`dropna()` removes rows with missing demand or temperature because OLS requires both values for every observation used in fitting.

We then divide the data into `temp_train` and `temp_test` using the same time boundary defined above.


**Create x and y for training and testing sets**

In [ ]:
temp_df = df[['timestamp', 'DMA1', 'Temperature']].dropna().copy()

# training data
temp_train = temp_df[temp_df['timestamp'] < test_start].copy()
# testing data
temp_test = temp_df[temp_df['timestamp'] >= test_start].copy()

# x,y train
X_train = temp_train[[💡]]
y_train = temp_train[💡]      # true values

# x,y test
X_test = temp_test[[💡]]
y_test = temp_test[💡]        # true values

In [ ]:
X_train.head()

In [ ]:
y_train.head()

### Before fitting the model let's explore if there is a relationship between temperature and demand?


In [ ]:
### scatter plot demand vs temperature

plt.scatter(💡, 💡, s=10, alpha=0.3)

plt.xlabel('Temperature (°C)')
plt.ylabel('Demand (L/s)')
plt.show()


# **Linear regression with scikit-learn**

We will use **scikit-learn (`sklearn`)**, a Python library that provides tools for building and evaluating machine learning models.

We will import:

- `LinearRegression` to fit an ordinary least squares (OLS) regression model.
- `mean_absolute_error` and `root_mean_squared_error` to evaluate how closely the model predictions match the observed values.

We will first fit the model using the training data, then use the fitted model to make predictions and evaluate its performance.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

## **Fit the OLS model in Python**

Scikit-learn separates model fitting into a few simple steps.

```python
model_temp = LinearRegression()
```
creates an empty linear regression model.

```python
model_temp.fit(X_train, y_train)
```
estimates the OLS coefficients from the training data. `X_train` contains the predictors and `y_train` contains the target.

After fitting:

- `model_temp.intercept_` contains $\hat\beta_0$;
- `model_temp.coef_` contains the estimated predictor coefficients.

The underscore at the end of names such as `coef_` is a scikit-learn convention indicating a value learned when the model was fit.


In [ ]:
# create the model
💡 = LinearRegression()
# fit the model
💡.💡(💡, 💡)

# get coefficents

print(f'Intercept: {model_temp.💡:.4f}')
print(f'Temperature coefficient: {model_temp.💡:.4f}')


### **Predict (training period)**

`predict(X_train)` applies the fitted equation to the predictor values in the test data.

The model coefficients are **not** estimated again. We use the fitted model to calculate predicted values for the observations used during training.

In [ ]:
# create a new column with predicted values

y_train_pred = 💡.💡(💡)

temp_train['Predicted'] = y_train_pred

In [ ]:
temp_train.head()

### Observed vs. predicted scatter plot for training data

In [ ]:
# plot

plt.scatter(temp_train['DMA1'], temp_train['Predicted'], s = 10, alpha = 0.5)
plt.xlabel('Observed demand (L/s)')
plt.ylabel('Predicted demand (L/s)')
plt.show()

### **Predict (testing period)**

`predict(X_test)` applies the fitted equation to the predictor values in the test data.

The model coefficients are **not** estimated again. We use the coefficients learned from the training period to predict observations that were not used for fitting.


In [ ]:
# add prediction to the temp_test dataframe

y_test_pred = 💡.💡(💡)
temp_test['Predicted'] = y_test_pred

temp_test.head()

In [ ]:
# predicted vs. observed for test data

plt.scatter(temp_test['DMA1'], temp_test['Predicted'], s = 10, alpha = 0.5)
plt.xlabel('Observed demand (L/s)')
plt.ylabel('Predicted demand (L/s)')
plt.show()

## **Forecast evaluation**

We will use two common error metrics.

### Mean Absolute Error

$$
MAE = \frac{1}{n}\sum_{t=1}^{n}|y_t-\hat y_t|
$$

MAE is the average magnitude of the prediction errors.

### Root Mean Squared Error

$$
RMSE = \sqrt{\frac{1}{n}\sum_{t=1}^{n}(y_t-\hat y_t)^2}
$$

RMSE gives more weight to large errors because the errors are squared.

Both metrics have the same units as demand, **L/s**, and lower values indicate smaller errors.


Both functions require two inputs, and **the order matters**:

1. **First:** the observed (true) values
2. **Second:** the predicted values from the model

```python
mean_absolute_error(observed, predicted)
root_mean_squared_error(observed, predicted)

In [ ]:
# evaluate performance training
mae_train = mean_absolute_error(💡, 💡)
rmse_train = root_mean_squared_error(💡, 💡)
print('MAE:', mae_train)
print('RMSE:', rmse_train)

In [ ]:
# evaluate performance test
mae_test_temp = mean_absolute_error(💡, 💡)
rmse_test_temp = root_mean_squared_error(💡, 💡)
print('MAE:', mae_test_temp)
print('RMSE:', rmse_test_temp)

### **Time-series view of Model 1**

A scatterplot shows the overall relationship, but a time-series plot shows whether the model reproduces the hourly pattern.

In [ ]:
# plot time series for the test period
plt.figure(figsize=(12, 4))
plt.plot(temp_test['timestamp'], temp_test['DMA1'], label='Observed')
plt.plot(temp_test['timestamp'], temp_test['Predicted'], label='Predicted')
plt.gcf().autofmt_xdate()
plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('Model 1: Temperature Model')
plt.legend()
plt.grid(True)
plt.show()


### Lets zoom in on the **first seven days of the test period** so individual daily cycles are visible.


In [ ]:
plot_history = 24*7


plt.figure(figsize=(12, 4))
plt.plot(temp_test['timestamp'].iloc[:plot_history], temp_test['DMA1'].iloc[:plot_history], label='Observed')
plt.plot(temp_test['timestamp'].iloc[:plot_history], temp_test['Predicted'].iloc[:plot_history], label='Predicted')
plt.gcf().autofmt_xdate()
plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('Model 1: Temperature Model - Seven-Day Zoom')
plt.legend()
plt.grid(True)
plt.show()


# **5. Model 2: OLS using lagged demand**

Temperature is an external predictor. Another source of information is the **history of the demand time series itself**.

For hourly data, two intuitive predictors are:

- $D_{t-1}$: demand one hour earlier;
- $D_{t-24}$: demand at the same hour one day earlier.

We will fit

$$
D_t = \beta_0 + \beta_1D_{t-1} + \beta_2D_{t-24} + \varepsilon_t
$$

and therefore predict

$$
\hat D_t = \hat\beta_0 + \hat\beta_1D_{t-1} + \hat\beta_2D_{t-24}
$$

### Creating lagged variables with `shift()`

Pandas `shift()` moves values down by a specified number of rows:

```python
df['lag_1'] = df['DMA1'].shift(1)
```

At time $t$, `lag_1` contains the DMA1 value from the previous row, $D_{t-1}$.

Similarly,

```python
df['lag_24'] = df['DMA1'].shift(24)
```

places the value from 24 rows earlier on the current row. Because we created a regular hourly series, this corresponds to $D_{t-24}$.

The first row of `lag_1` must be missing because there is no earlier observation. The first 24 rows of `lag_24` must be missing because 24 hours of history are not yet available.


In [ ]:
df['lag_1'] = df['DMA1'].shift(💡)
df['lag_24'] = df['DMA1'].shift(💡)

df[['timestamp', 'DMA1', 'lag_1', 'lag_24']].head(💡)


## **Prepare and fit the lag model**

Here `predictors` is simply a Python list containing the column names we want to use as inputs.

```python
predictors = ['lag_1', 'lag_24']
```

We can then select both columns at once with

```python
lag_train[predictors]
```

This is useful because the same code works whether the model contains two predictors or twenty predictors.


In [ ]:
# create a copy for the lag model (drop NA)

lag_df = df[['timestamp', 'DMA1', 'lag_1', 'lag_24']].dropna().copy()

lag_train = lag_df[lag_df['timestamp'] < test_start].copy()
lag_test = lag_df[lag_df['timestamp'] >= test_start].copy()

predictors = ['lag_1', 'lag_24']

# x,y train
X_train = lag_train[💡]
y_train = lag_train['DMA1']       # true values

# x,y test
X_test = lag_test[💡]
y_test = lag_test['DMA1']        # true values

# create model
💡 = LinearRegression()

# fit model
💡.💡(💡, 💡)

print('Intercept:', model_lag.intercept_)
print(pd.Series(model_lag.coef_, index=predictors))


## **Predict and evaluate Model 2**

The output of `predict()` is an array containing one predicted demand value for each row in `lag_test`. We store those values as a new column so the predictions remain aligned with their timestamps and observed values.


In [ ]:
y_test_pred = 💡.💡(💡)

lag_test['Predicted'] = y_test_pred

lag_test.head()


### **Evaluate lag model**

In [ ]:
# evaluate performance test
mae_test_lag = mean_absolute_error(y_test, y_test_pred)
rmse_test_lag = root_mean_squared_error(y_test, y_test_pred)
print('MAE:', mae_test_lag)
print('RMSE:', rmse_test_lag)


### **Time-series view of Model 2**

Use the same seven-day window as Model 1. Keeping the plotting window the same makes visual comparison between models easier.


In [ ]:
# plot time series for the test period
plt.figure(figsize=(12, 4))

plt.plot(lag_test['timestamp'], lag_test['DMA1'], label='Observed')
plt.plot(lag_test['timestamp'], lag_test['Predicted'], label='Predicted')
plt.gcf().autofmt_xdate()
plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('Model 2: Lag Model')
plt.legend()
plt.grid(True)
plt.show()


# **6. Model 3: Add time of day as a categorical predictor**

Water demand depends strongly on **time of day**. Morning demand may systematically differ from nighttime demand even at the same temperature.

Hour of day is categorical in this model. We do **not** want to assume that changing from hour 1 to hour 2 has exactly the same linear effect as changing from hour 12 to hour 13.

We begin with the temperature model and add time of day:

$$
D_t = \beta_0 + \beta_TT_t + \sum_{h=1}^{23}\gamma_h I(H_t=h) + \varepsilon_t
$$

where:

- $T_t$ is temperature;
- $H_t$ is the hour of day;
- $I(H_t=h)$ is an **indicator (dummy) variable** that equals 1 when the observation occurs during hour $h$ and 0 otherwise;
- $\gamma_h$ measures how expected demand at hour $h$ differs from the reference hour.

There are 24 possible hours, but only 23 dummy variables are included. One category is omitted and becomes the **reference category**. Here, hour 0 (midnight) is the reference.

This avoids including redundant information in the regression.


### Extract hour of day

Because `timestamp` is a datetime variable, pandas provides datetime properties through `.dt`.

```python
df['timestamp'].dt.hour
```

returns an integer from 0 to 23 for each timestamp.

We store it in a new column called `hour`.


In [ ]:
df['hour'] = df['timestamp'].dt.hour

df[['timestamp', 'DMA1', 'Temperature', 'hour']].head(26)


In [ ]:
df.head()

## **Convert the categorical variable to dummy variables**

Scikit-learn's linear regression expects numerical inputs. `pd.get_dummies()` converts the hour categories into 0/1 columns.

```python
pd.get_dummies(..., columns=['hour'])
```

creates columns such as `hour_1`, `hour_2`, ..., `hour_23`.

```python
drop_first=True
```

omits the first category, hour 0, which becomes the reference category.

```python
dtype=int
```

stores the dummy variables as 0 and 1 rather than `False` and `True`.

After this transformation, a row recorded at 8:00 has `hour_8 = 1` and the other hour dummy variables equal 0.


In [ ]:
time_df = df[['timestamp', 'DMA1', 'Temperature', 'hour']].dropna().copy()

time_df = pd.get_dummies(
    time_df,
    columns=['hour'],
    prefix='hour',
    drop_first=💡,
    dtype=int
)

time_df.head()


### Fit the temperature + time-of-day model



In [ ]:
# training and testing data
time_train = time_df[time_df['timestamp'] < test_start].copy()
time_test = time_df[time_df['timestamp'] >= test_start].copy()

# x,y training
X_train = time_train.iloc[:,💡]
y_train = time_train['DMA1']

# x,y testing
X_test = time_test.iloc[:,💡]
y_test = time_test['DMA1']




In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
# fit model

model_time = LinearRegression()
model_time.fit(X_train, y_train)

print('Intercept:', model_time.intercept_)
print('Temperature coefficient:', model_time.coef_[0])

### Interpret the hour coefficients

Because hour 0 is the reference category, each hour coefficient is interpreted **relative to midnight**, holding temperature constant.

For example, if the coefficient for `hour_8` is positive, the model estimates higher demand at 8:00 than at midnight for the same temperature.

The coefficients do not need to increase smoothly with hour. Each hour receives its own estimated effect.


In [ ]:
model_time.coef_[💡]


## **Predict and evaluate Model 3**


In [ ]:
# predict

y_test_pred = model_time.predict(X_test)

time_test['Predicted'] = y_test_pred

time_test.head()


### **Evaluate time model**

In [ ]:
# evaluate performance test
mae_test_time = mean_absolute_error(y_test, y_test_pred)
rmse_test_time = root_mean_squared_error(y_test, y_test_pred)
print('MAE:', mae_test_time)
print('RMSE:', rmse_test_time)


## **Time-series view of Model 3**

The time-of-day dummy variables allow the model to represent a repeating daily shape. Compare this plot with the temperature-only model.


In [ ]:
# plot time series for the test period
plt.figure(figsize=(12, 4))

plt.plot(time_test['timestamp'], time_test['DMA1'], label='Observed')
plt.plot(time_test['timestamp'], time_test['Predicted'], label='Predicted')
plt.gcf().autofmt_xdate()
plt.xlabel('Time')
plt.ylabel('Demand (L/s)')
plt.title('Model 3: Time Model')
plt.legend()
plt.grid(True)
plt.show()


# **7. Compare the three regression models**

The models use different sources of information:

| Model | Predictors | Type of information |
|---|---|---|
| 1 | Temperature | External numerical predictor |
| 2 | Lag 1 + lag 24 | Historical target values |
| 3 | Temperature + hour of day | Numerical + categorical predictors |

A lower error does not automatically mean that a model is preferable in every application. Also consider what information would actually be available when a forecast is made.


In [ ]:
comparison = pd.DataFrame({
    'Model': [
        'Temperature',
        'Lag 1 + Lag 24',
        'Temperature + Hour of Day'
    ],
    'MAE': [
        mae_test_temp,
        mae_test_lag,
        mae_test_time
    ],
    'RMSE': [
        rmse_test_temp,
        rmse_test_lag,
        rmse_test_time
    ]
})

comparison


# **8. Exercises: combine the predictors**

You now have all the pieces needed to build a model that combines the different sources of information.

Try a model of the form

$$
D_t = \beta_0
+ \beta_TT_t
+ \beta_1D_{t-1}
+ \beta_{24}D_{t-24}
+ \sum_{h=1}^{23}\gamma_h I(H_t=h)
+ \varepsilon_t
$$

### Tasks

1. Create a DataFrame containing demand, temperature, `lag_1`, `lag_24`, and hour of day.
2. Remove rows with missing required values.
3. Convert hour of day to dummy variables.
4. Split the data into training and testing periods using `test_start`.
5. Fit the combined OLS model.
6. Predict demand during the test period.
7. Calculate MAE and RMSE.
8. Create a seven-day time-series plot of observed and predicted demand.
9. Compare the combined model with Models 1–3.

**Question:** Does adding more predictors necessarily improve predictions on the test data?


In [ ]:
# Your code here


# **10. Key Python ideas from this notebook**

### `.copy()`
Creates a separate DataFrame that we can safely modify.

```python
new_df = old_df[condition].copy()
```

### `.shift(k)`
Moves a series by `k` rows and is useful for creating lagged predictors.

```python
df['lag_24'] = df['DMA1'].shift(24)
```

### `.dropna()`
Removes rows containing missing values in the selected DataFrame.

```python
model_df = df[['DMA1', 'lag_24']].dropna()
```

### `.dt.hour`
Extracts hour of day from a datetime column.

```python
df['hour'] = df['timestamp'].dt.hour
```

### `pd.get_dummies()`
Converts a categorical variable into numerical 0/1 indicator variables.

```python
pd.get_dummies(df, columns=['hour'], drop_first=True)
```

### `LinearRegression()`
Creates a linear regression model.

### `.fit(X, y)`
Estimates the model coefficients using predictor matrix `X` and target `y`.

### `.predict(X)`
Uses the fitted coefficients to calculate predicted values for new predictor data.

### Why is `X`  selected with double brackets?

Scikit-learn expects the predictors to be a two-dimensional table, even when there is only one predictor:

```python
X = df[['Temperature']]
```

The target is one-dimensional:

```python
y = df['DMA1']
```

Keeping this distinction in mind helps avoid many common scikit-learn errors.


## 12. Final discussion

1. What pattern could the temperature-only model capture? What pattern could it not capture?
2. Why are `lag_1` and `lag_24` potentially useful for hourly water demand?
3. Why did we treat hour of day as categorical rather than simply using values 0–23 as one numerical predictor?
4. Which model produced the lowest test error?
5. Which model best reproduced the daily pattern in the seven-day plots?
6. What information would need to be known at the time a real forecast is made?
7. What additional predictors might be useful for water-demand forecasting?
7. Why must we be careful not to use predictor values that occur after the forecast origin?
